In [ ]:
#1 Notebook description:

# this notebook is used to evaluate the market of assets and find potential assets to invest in
# with the best risk/reward characteristics. This notebook is intented to analyze a broad group of market assets
# and does NOT focus on any particular assets. Computing data for a large number of assets is computationally expensive and thus
# the analysis is relegated to other notebooks
# this notebook now also replaces the old 'Market Study - Statistical' notebook through the preset options below

In [ ]:
#2 Load libraries
import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
# Load libraries
import sys
import os
project_path = os.getcwd()
while not os.path.exists(os.path.join(project_path, "pyproject.toml")):
    parent = os.path.dirname(project_path)
    if parent == project_path:
        raise FileNotFoundError("Could not locate the project root from the current working directory.")
    project_path = parent
if project_path not in sys.path:
    sys.path.append(project_path)
from Quantapp.visualization import Plotter
from Quantapp.analytics import Rolling, RiskRelativeAnalytics, SignalLabels, SeriesTransforms
from Quantapp.analytics.series_utils import calculate_zscore
from Quantapp.data import MacroDataClient
from Quantapp.data import MarketDataClient

import numpy as np
import json
import pandas as pd
import yfinance as yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.subplots as sp
import plotly.graph_objects as go
import pandas as pd
import holidays
import plotly.express as px
import concurrent.futures
from plotly.subplots import make_subplots
import plotly.graph_objects as go

#shut down warnings
import warnings
warnings.filterwarnings("ignore")
pio.templates.default = 'plotly_dark'



rolling = Rolling()
risk_relative_analytics = RiskRelativeAnalytics()
signal_labels = SignalLabels()
series_transforms = SeriesTransforms()
qp = Plotter()
qe = MacroDataClient()
market_data = MarketDataClient()
                

In [ ]:
#3 Parameters
time_frame_week = 7
time_frame_short = 21
time_frame_mid   = 50
time_frame_long = 200
interval = '1d'
period     = '10y'
risk_free_rate = 0.02 / 252  # Annualized risk-free rate divided by trading days
benchmark = 'SPY'
mode='standard'
analysis_preset = 'broad'  # supported: 'broad', 'statistical'
preset_config = {
    'broad': {'asset_class': 'broad', 'sector': 'all'},
    'statistical': {'asset_class': 'equity', 'sector': 'financials'},
}
asset_class = preset_config[analysis_preset]['asset_class']
sector = preset_config[analysis_preset]['sector']
# Optional manual overrides:
# asset_class = 'equity'
# sector supported values: 'all', 'energy', 'materials', 'industrials', 'consumer_discretionary', 'consumer_staples', 'healthcare', 'financials', 'information_technology', 'communication_services', 'utilities', 'real_estate'
# sector = 'healthcare'

In [ ]:
#3A Historical Treasury Yields
treasury_start_date = None  # Pull the full available FRED history for each maturity.
treasury_maturity_order = ['1M', '3M', '6M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
broad_treasury_yields = pd.DataFrame()
ordered_maturities = []
broad_treasury_average_yield = pd.Series(dtype=float)

fred_key = os.getenv('FRED_API_KEY')
if not fred_key and os.path.exists('.env'):
    with open('.env', 'r', encoding='utf-8') as env_file:
        for line in env_file:
            if line.startswith('FRED_API_KEY='):
                fred_key = line.strip().split('=', 1)[1]
                break

if not fred_key:
    print('Skipping Historical Treasury Yields: FRED_API_KEY is not configured.')
else:
    treasury_client = MacroDataClient(fred_key=fred_key)
    broad_treasury_yields = treasury_client.get_historical_treasury_yields(start_date=treasury_start_date).sort_index().dropna(how='all')
    ordered_maturities = [maturity for maturity in treasury_maturity_order if maturity in broad_treasury_yields.columns]

    if not ordered_maturities:
        print('No Treasury yield history is available for plotting.')
    else:
        treasury_history_fig = go.Figure()
        for maturity in ordered_maturities:
            series = broad_treasury_yields[maturity].dropna()
            if series.empty:
                continue
            treasury_history_fig.add_trace(
                go.Scatter(
                    x=series.index,
                    y=series,
                    mode='lines',
                    name=maturity,
                    line=dict(width=1.6),
                )
            )

        treasury_history_fig.update_layout(
            title='Historical Treasury Yields',
            template='plotly_dark',
            height=700,
            hovermode='x unified',
            legend_title_text='Maturity',
            margin=dict(l=40, r=40, t=90, b=40),
        )
        treasury_history_fig.update_xaxes(title_text='Date')
        treasury_history_fig.update_yaxes(title_text='Yield (%)')
        treasury_history_fig.show(config={'responsive': True})

        broad_treasury_average_yield = broad_treasury_yields[ordered_maturities].mean(axis=1).dropna()
        if broad_treasury_average_yield.empty:
            print('No Treasury average-yield history is available for plotting.')
        else:
            average_yield_fig = go.Figure()
            average_yield_fig.add_trace(
                go.Scatter(
                    x=broad_treasury_average_yield.index,
                    y=broad_treasury_average_yield,
                    mode='lines',
                    name='Average Yield Level',
                    line=dict(color='#f59e0b', width=2.5),
                    hovertemplate='Date=%{x|%Y-%m-%d}<br>Average yield=%{y:.2f}%<extra></extra>',
                )
            )
            average_yield_fig.add_hline(
                y=float(broad_treasury_average_yield.mean()),
                line_dash='dot',
                line_color='rgba(245, 158, 11, 0.55)',
                annotation_text='Full-Sample Mean',
                annotation_position='top left',
            )
            average_yield_fig.update_layout(
                title='Historical Average Treasury Yield Across Maturities',
                template='plotly_dark',
                height=500,
                hovermode='x unified',
                legend_title_text='Series',
                margin=dict(l=40, r=40, t=90, b=40),
            )
            average_yield_fig.update_xaxes(title_text='Date')
            average_yield_fig.update_yaxes(title_text='Average Yield (%)')
            average_yield_fig.show(config={'responsive': True})


In [ ]:
#3B Treasury Yield Curve Animation
if broad_treasury_yields.empty or not ordered_maturities:
    print('Run the Historical Treasury Yields cell first to load Treasury data for the animation.')
else:
    treasury_curve_animation_data = broad_treasury_yields[ordered_maturities].ffill().dropna(how='all')
    monthly_frame_dates = (
        treasury_curve_animation_data.groupby(treasury_curve_animation_data.index.to_period('M'))
        .apply(lambda frame: frame.index[-1])
        .tolist()
    )
    treasury_curve_animation_data = treasury_curve_animation_data.loc[monthly_frame_dates]

    if treasury_curve_animation_data.empty:
        print('No Treasury yield curve snapshots are available for animation.')
    else:
        curve_min = float(treasury_curve_animation_data.min().min())
        curve_max = float(treasury_curve_animation_data.max().max())
        curve_padding = max(0.25, (curve_max - curve_min) * 0.08) if curve_max > curve_min else 0.5

        def _curve_trace(frame_date, values):
            return go.Scatter(
                x=ordered_maturities,
                y=values.tolist(),
                mode='lines+markers',
                line=dict(color='#60a5fa', width=3),
                marker=dict(color='#f8fafc', size=9, line=dict(color='#60a5fa', width=1.5)),
                fill='tozeroy',
                fillcolor='rgba(96, 165, 250, 0.18)',
                hovertemplate='Maturity=%{x}<br>Yield=%{y:.2f}%<extra></extra>',
                name=frame_date.strftime('%Y-%m-%d'),
            )

        def _average_level_trace(average_yield):
            return go.Scatter(
                x=ordered_maturities,
                y=[average_yield] * len(ordered_maturities),
                mode='lines',
                name='Average Curve Level',
                line=dict(color='#f59e0b', width=2, dash='dot'),
                hovertemplate='Average curve level=%{y:.2f}%<extra></extra>',
            )

        initial_curve_date = treasury_curve_animation_data.index[0]
        initial_curve_values = treasury_curve_animation_data.iloc[0]
        initial_curve_average = float(initial_curve_values.mean())
        animation_frames = []
        slider_steps = []

        for frame_date, curve_values in treasury_curve_animation_data.iterrows():
            frame_name = frame_date.strftime('%Y-%m-%d')
            curve_average = float(curve_values.mean())
            animation_frames.append(
                go.Frame(
                    name=frame_name,
                    data=[
                        _curve_trace(frame_date, curve_values),
                        _average_level_trace(curve_average),
                    ],
                )
            )
            slider_steps.append(
                dict(
                    label=frame_date.strftime('%Y-%m'),
                    method='animate',
                    args=[
                        [frame_name],
                        {
                            'mode': 'immediate',
                            'frame': {'duration': 0, 'redraw': True},
                            'transition': {'duration': 0},
                        },
                    ],
                )
            )

        treasury_curve_animation_fig = go.Figure(
            data=[
                _curve_trace(initial_curve_date, initial_curve_values),
                _average_level_trace(initial_curve_average),
            ],
            frames=animation_frames,
        )
        treasury_curve_animation_fig.update_layout(
            title='Treasury Yield Curve Animation',
            template='plotly_dark',
            height=650,
            hovermode='closest',
            margin=dict(l=40, r=40, t=90, b=110),
            xaxis=dict(
                title='Maturity',
                categoryorder='array',
                categoryarray=ordered_maturities,
            ),
            yaxis=dict(
                title='Yield (%)',
                range=[curve_min - curve_padding, curve_max + curve_padding],
            ),
            updatemenus=[
                dict(
                    type='buttons',
                    direction='left',
                    showactive=False,
                    x=0.01,
                    y=1.16,
                    xanchor='left',
                    yanchor='top',
                    buttons=[
                        dict(
                            label='Play',
                            method='animate',
                            args=[
                                None,
                                {
                                    'fromcurrent': True,
                                    'frame': {'duration': 220, 'redraw': True},
                                    'transition': {'duration': 140},
                                },
                            ],
                        ),
                        dict(
                            label='Pause',
                            method='animate',
                            args=[
                                [None],
                                {
                                    'mode': 'immediate',
                                    'frame': {'duration': 0, 'redraw': False},
                                    'transition': {'duration': 0},
                                },
                            ],
                        ),
                    ],
                )
            ],
            sliders=[
                dict(
                    active=0,
                    currentvalue={'prefix': 'Curve Date: '},
                    pad={'t': 55},
                    len=0.95,
                    x=0.03,
                    xanchor='left',
                    y=0,
                    yanchor='top',
                    steps=slider_steps,
                )
            ],
        )
        treasury_curve_animation_fig.show(config={'responsive': True})


In [ ]:
#3C Treasury Slopes and Belly
if broad_treasury_yields.empty or not ordered_maturities:
    print('Run the Historical Treasury Yields cell first to load Treasury data for the slope and belly plots.')
else:
    treasury_curve_filled = broad_treasury_yields[ordered_maturities].ffill().dropna(how='all')
    slope_series_map = {}
    slope_pairs = [
        ('10Y - 3M', '10Y', '3M'),
        ('10Y - 2Y', '10Y', '2Y'),
        ('5Y - 2Y', '5Y', '2Y'),
        ('30Y - 10Y', '30Y', '10Y'),
    ]

    for label, long_maturity, short_maturity in slope_pairs:
        if {long_maturity, short_maturity}.issubset(treasury_curve_filled.columns):
            slope_series_map[label] = (
                treasury_curve_filled[long_maturity] - treasury_curve_filled[short_maturity]
            ).dropna()

    belly_series_map = {}
    if {'2Y', '5Y', '10Y'}.issubset(treasury_curve_filled.columns):
        belly_series_map['5Y Belly vs Avg(2Y,10Y)'] = (
            treasury_curve_filled['5Y'] - (
                (treasury_curve_filled['2Y'] + treasury_curve_filled['10Y']) / 2
            )
        ).dropna()

    slope_df = pd.DataFrame(slope_series_map).dropna(how='all')
    belly_df = pd.DataFrame(belly_series_map).dropna(how='all')

    if slope_df.empty and belly_df.empty:
        print('No Treasury slope or belly series are available for plotting.')
    else:
        treasury_slope_fig = make_subplots(
            rows=2,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.1,
            subplot_titles=(
                'Historical Treasury Slopes',
                'Historical Belly / Curvature',
            ),
        )

        slope_palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
        for idx, column in enumerate(slope_df.columns):
            treasury_slope_fig.add_trace(
                go.Scatter(
                    x=slope_df.index,
                    y=slope_df[column],
                    mode='lines',
                    name=column,
                    line=dict(color=slope_palette[idx % len(slope_palette)], width=2.2),
                ),
                row=1,
                col=1,
            )

        belly_palette = ['#f59e0b', '#22c55e', '#f97316']
        for idx, column in enumerate(belly_df.columns):
            treasury_slope_fig.add_trace(
                go.Scatter(
                    x=belly_df.index,
                    y=belly_df[column],
                    mode='lines',
                    name=column,
                    line=dict(color=belly_palette[idx % len(belly_palette)], width=2.4),
                ),
                row=2,
                col=1,
            )

        treasury_slope_fig.add_hline(y=0, line_dash='dot', line_color='rgba(226, 232, 240, 0.45)', row=1, col=1)
        treasury_slope_fig.add_hline(y=0, line_dash='dot', line_color='rgba(226, 232, 240, 0.45)', row=2, col=1)
        treasury_slope_fig.update_yaxes(title_text='Spread (percentage points)', row=1, col=1)
        treasury_slope_fig.update_yaxes(title_text='Belly Spread (%)', row=2, col=1)
        treasury_slope_fig.update_xaxes(title_text='Date', row=2, col=1)
        treasury_slope_fig.update_layout(
            title='Treasury Slopes and Belly Over Time',
            template='plotly_dark',
            height=850,
            hovermode='x unified',
            legend_title_text='Series',
            margin=dict(l=40, r=40, t=90, b=40),
        )
        treasury_slope_fig.show(config={'responsive': True})


In [ ]:
#4 Broad market reference: cumulative returns, rolling Sharpe z-scores, and Sharpe spread z-scores
broad_market_ticker_map = {
    'Global Equities': 'ACWI',
    'U.S. Total Equity': 'VTI',
    'U.S. Large Cap': 'SPY',
    'International ex-U.S.': 'VXUS',
    'Emerging Markets': 'IEMG',
    'U.S. Aggregate Bonds': 'AGG',
    'Global Bonds': 'BNDW',
    'Broad Commodities': 'DBC',
    'Gold': 'GLD',
    'Crypto (Bitcoin)': 'BTC-USD',
    'U.S. Real Estate': 'VNQ',
    'U.S. Dollar Index': 'DX-Y.NYB'
}

# Match the default Risk Analysis history depth for block #12 comparisons.
broad_market_reference_period = '20y'

broad_market_reference = {
    name: yf.Ticker(ticker).history(period=broad_market_reference_period, interval=interval)
    for name, ticker in broad_market_ticker_map.items()
}
broad_market_display_name_map = {
    name: f"{broad_market_ticker_map.get(name, 'N/A')} | {name}"
    for name in broad_market_ticker_map
}

def _normalize_market_index(data):
    normalized = data.copy()
    if not isinstance(normalized.index, pd.DatetimeIndex):
        return normalized
    index = normalized.index
    if index.tz is not None:
        index = index.tz_convert('UTC').tz_localize(None)
    normalized.index = index.normalize()
    return normalized.sort_index()

broad_market_reference = {
    name: _normalize_market_index(df)
    for name, df in broad_market_reference.items()
}

broad_market_close_map = {
    name: df['Close'].dropna().sort_index()
    for name, df in broad_market_reference.items()
    if not df.empty and 'Close' in df
}
broad_market_close = pd.DataFrame(broad_market_close_map).sort_index()

def _cumulative_return_series(series):
    valid = series.dropna()
    if valid.empty:
        return series
    return valid.div(valid.iloc[0]).sub(1).mul(100)

def plot_cumulative_return_overlay(price_data, title, default_label='1y', display_name_map=None):
    plot_data = price_data.dropna(how='all')
    if plot_data.empty or plot_data.shape[1] == 0:
        print(f'No data available for {title}.')
        return

    plot_columns = plot_data.columns.tolist()
    end_date = plot_data.index.max()
    min_date = plot_data.index.min()
    timeframe_offsets = [
        ('3mo', pd.DateOffset(months=3)),
        ('6mo', pd.DateOffset(months=6)),
        ('9mo', pd.DateOffset(months=9)),
        ('1y', pd.DateOffset(years=1)),
        ('2y', pd.DateOffset(years=2)),
    ]

    cumulative_frames = {}
    for label, offset in timeframe_offsets:
        start_date = max(min_date, end_date - offset)
        window_prices = plot_data.loc[start_date:end_date]
        cumulative_frames[label] = window_prices.apply(_cumulative_return_series).reindex(window_prices.index)

    available_labels = [
        label for label, frame in cumulative_frames.items()
        if not frame.dropna(how='all').empty
    ]
    if not available_labels:
        print(f'No data available for {title}.')
        return

    if default_label not in available_labels:
        default_label = available_labels[0]

    default_frame = cumulative_frames[default_label]
    fig = go.Figure()
    for column in plot_columns:
        display_name = display_name_map.get(column, column) if display_name_map else column
        fig.add_trace(
            go.Scatter(
                x=default_frame.index,
                y=default_frame[column],
                mode='lines',
                name=display_name,
                showlegend=True
            )
        )

    buttons = []
    for label in available_labels:
        frame = cumulative_frames[label]
        x_values = [frame.index.tolist()] * len(plot_columns)
        y_values = [frame[column].tolist() for column in plot_columns]
        buttons.append(
            dict(
                label=label,
                method='update',
                args=[
                    {'x': x_values, 'y': y_values},
                    {'title': f'{title} ({label})'}
                ]
            )
        )

    fig.update_layout(
        title=f'{title} ({default_label})',
        height=700,
        template='plotly_dark',
        autosize=True,
        margin=dict(l=40, r=40, t=90, b=40),
        updatemenus=[
            dict(
                buttons=buttons,
                direction='down',
                showactive=True,
                active=available_labels.index(default_label),
                x=0.01,
                y=1.18,
                xanchor='left',
                yanchor='top'
            )
        ],
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
    )
    fig.update_xaxes(title_text='Date')
    fig.update_yaxes(title_text='Cumulative Return (%)')
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.show(config={'responsive': True})

def plot_reference_dataframe(data, title, yaxis_title, add_zero_line=False, sigma_lines=None, overlay_series=False, display_name_map=None, style=None):
    if data.empty:
        print(f'No data available for {title}.')
        return

    plot_data = data.dropna(how='all')
    if plot_data.empty or plot_data.shape[1] == 0:
        print(f'No data available for {title}.')
        return

    plot_columns = plot_data.columns.tolist()
    num_rows = 1 if overlay_series else len(plot_columns)
    end_date = plot_data.index.max()
    min_date = plot_data.index.min()
    zoom_ranges = [
        ('3mo', pd.DateOffset(months=3)),
        ('6mo', pd.DateOffset(months=6)),
        ('9mo', pd.DateOffset(months=9)),
        ('1y', pd.DateOffset(years=1)),
        ('2y', pd.DateOffset(years=2)),
    ]
    default_label = '2y'
    default_start = max(min_date, end_date - dict(zoom_ranges)[default_label])

    def _xaxis_range_update(start_date, end_date):
        range_update = {}
        for axis_idx in range(1, num_rows + 1):
            axis_name = 'xaxis' if axis_idx == 1 else f'xaxis{axis_idx}'
            range_update[f'{axis_name}.range'] = [start_date, end_date]
        return range_update

    zoom_buttons = []
    for label, offset in zoom_ranges:
        start_date = max(min_date, end_date - offset)
        zoom_buttons.append(
            dict(
                label=label,
                method='relayout',
                args=[_xaxis_range_update(start_date, end_date)]
            )
        )

    subplot_spacing = 0.0 if overlay_series else min(0.03, 0.8 / max(num_rows, 2)) if num_rows > 1 else 0.0
    fig = make_subplots(
        rows=num_rows,
        cols=1,
        shared_xaxes=not overlay_series,
        vertical_spacing=subplot_spacing,
        subplot_titles=None if overlay_series else [display_name_map.get(column, column) if display_name_map else column for column in plot_columns]
    )
    for annotation in fig.layout.annotations:
        annotation.font = dict(size=10, color='rgba(220, 220, 220, 0.90)')

    for row_idx, column in enumerate(plot_columns, start=1):
        target_row = 1 if overlay_series else row_idx
        display_name = display_name_map.get(column, column) if display_name_map else column
        fig.add_trace(
            go.Scatter(
                x=plot_data.index,
                y=plot_data[column],
                mode='lines',
                name=display_name,
                showlegend=overlay_series
            ),
            row=target_row,
            col=1
        )

        if not overlay_series:
            if style:
                _apply_reference_panel_style(fig, row_idx, 1, style, add_zero_line=add_zero_line, sigma_lines=sigma_lines)
            else:
                if add_zero_line:
                    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=row_idx, col=1)
                if sigma_lines:
                    for sigma in sigma_lines:
                        fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=1)
                        fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=1)

        if not overlay_series:
            fig.update_yaxes(title_text=yaxis_title, row=row_idx, col=1)

    if overlay_series:
        if add_zero_line:
            fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)
        if sigma_lines:
            for sigma in sigma_lines:
                fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=1, col=1)
                fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=1, col=1)
        fig.update_yaxes(title_text=yaxis_title, row=1, col=1)

    fig.update_layout(
        title=title,
        height=700 if overlay_series else max(420, 220 * num_rows + 120),
        template='plotly_dark',
        autosize=True,
        margin=dict(l=40, r=40, t=90, b=40),
        updatemenus=[
            dict(
                buttons=zoom_buttons,
                direction='down',
                showactive=True,
                active=4,
                x=0.01,
                y=1.18,
                xanchor='left',
                yanchor='top'
            )
        ]
    )
    if overlay_series:
        fig.update_layout(legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0))
    for row_idx in range(1, num_rows + 1):
        fig.update_xaxes(range=[default_start, end_date], row=row_idx, col=1)
    fig.update_xaxes(title_text='Date', row=num_rows, col=1)
    fig.show(config={'responsive': True})

def _add_reference_zone(fig, row, col, y0, y1, fillcolor, opacity, label=None, font_color='rgba(235, 235, 235, 0.95)', x=0.5, font_size=14):
    fig.add_hrect(
        y0=y0,
        y1=y1,
        fillcolor=fillcolor,
        opacity=opacity,
        line_width=0,
        layer='below',
        row=row,
        col=col
    )
    if label:
        fig.add_annotation(
            x=x,
            y=(y0 + y1) / 2,
            xref='x domain',
            yref='y',
            text=label,
            showarrow=False,
            xanchor='center',
            yanchor='middle',
            font=dict(color=font_color, size=font_size),
            row=row,
            col=col
        )

def _add_std_level_annotations(fig, row, col, levels):
    labels = [(0, 'Mean')]
    for level in levels:
        labels.append((level, f'+{level} SD'))
        labels.append((-level, f'-{level} SD'))

    for y_value, label in labels:
        fig.add_annotation(
            x=0.985,
            y=y_value,
            xref='x domain',
            yref='y',
            text=label,
            showarrow=False,
            xanchor='left',
            yanchor='middle',
            font=dict(color='rgba(220, 220, 220, 0.90)', size=11),
            align='left',
            row=row,
            col=col
        )

def _apply_reference_panel_style(fig, row, col, style, add_zero_line=False, sigma_lines=None):
    if style == 'risk_detail':
        _add_reference_zone(fig, row, col, -1, 1, 'rgba(211, 211, 211, 0.18)', 1.0, label='Neutral')
        fig.add_annotation(
            x=0.5,
            y=-0.55,
            xref='x domain',
            yref='y',
            text='Bullish Neutral (on the way up)',
            showarrow=False,
            xanchor='center',
            yanchor='middle',
            font=dict(color='rgba(235, 235, 235, 0.90)', size=12),
            row=row,
            col=col
        )
        fig.add_annotation(
            x=0.5,
            y=0.55,
            xref='x domain',
            yref='y',
            text='Bearish Neutral (on the way down)',
            showarrow=False,
            xanchor='center',
            yanchor='middle',
            font=dict(color='rgba(235, 235, 235, 0.90)', size=12),
            row=row,
            col=col
        )
        _add_reference_zone(fig, row, col, -2, -1, 'rgba(0, 128, 0, 0.30)', 1.0, label='Accumulate', font_color='rgba(235, 255, 235, 0.95)')
        _add_reference_zone(fig, row, col, 1, 2, 'rgba(180, 0, 0, 0.30)', 1.0, label='Liquidate', font_color='rgba(255, 235, 235, 0.95)')
        levels = sigma_lines if sigma_lines is not None else [1, 2, 3]
        for sigma in levels:
            fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(220, 220, 220, 0.55)', line_width=1, row=row, col=col)
            fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(220, 220, 220, 0.55)', line_width=1, row=row, col=col)
        if add_zero_line:
            fig.add_hline(y=0, line_dash='dash', line_color='rgba(160, 160, 160, 0.60)', line_width=1, row=row, col=col)

    elif style == 'risk_summary':
        _add_reference_zone(fig, row, col, -2, -1.5, 'rgba(180, 0, 0, 0.40)', 1.0, label='Liquidate', font_color='rgba(255, 235, 235, 0.95)')
        _add_reference_zone(fig, row, col, 1.5, 2, 'rgba(0, 128, 0, 0.55)', 1.0, label='Accumulate', font_color='rgba(235, 255, 235, 0.95)')
        fig.add_hline(y=0, line_color='rgba(255, 255, 255, 0.80)', line_width=1, row=row, col=col)
        levels = sigma_lines if sigma_lines is not None else [0.5, 1, 1.5, 2]
        for sigma in levels:
            fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(220, 220, 220, 0.55)', line_width=1, row=row, col=col)
            fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(220, 220, 220, 0.55)', line_width=1, row=row, col=col)
        _add_std_level_annotations(fig, row, col, levels)

def plot_reference_dataframes_side_by_side(
    left_data,
    right_data,
    title,
    left_yaxis_title,
    right_yaxis_title,
    left_add_zero_line=False,
    right_add_zero_line=False,
    left_sigma_lines=None,
    right_sigma_lines=None,
    left_style=None,
    right_style=None,
    display_name_map=None
):
    left_plot_data = left_data.dropna(how='all') if left_data is not None else pd.DataFrame()
    right_plot_data = right_data.dropna(how='all') if right_data is not None else pd.DataFrame()

    left_is_empty = left_plot_data.empty or left_plot_data.shape[1] == 0
    right_is_empty = right_plot_data.empty or right_plot_data.shape[1] == 0
    if left_is_empty and right_is_empty:
        print(f'No data available for {title}.')
        return

    left_columns = [] if left_is_empty else left_plot_data.columns.tolist()
    right_columns = [] if right_is_empty else right_plot_data.columns.tolist()
    plot_columns = list(dict.fromkeys(left_columns + right_columns))
    num_rows = len(plot_columns)

    date_indexes = []
    if not left_is_empty:
        date_indexes.append(left_plot_data.index)
    if not right_is_empty:
        date_indexes.append(right_plot_data.index)

    end_date = max(index.max() for index in date_indexes)
    min_date = min(index.min() for index in date_indexes)
    zoom_ranges = [
        ('3mo', pd.DateOffset(months=3)),
        ('6mo', pd.DateOffset(months=6)),
        ('9mo', pd.DateOffset(months=9)),
        ('1y', pd.DateOffset(years=1)),
        ('2y', pd.DateOffset(years=2)),
    ]
    default_label = '2y'
    default_start = max(min_date, end_date - dict(zoom_ranges)[default_label])

    root_left_axis = 'xaxis' if num_rows == 1 else f'xaxis{((num_rows - 1) * 2) + 1}'
    root_right_axis = 'xaxis2' if num_rows == 1 else f'xaxis{((num_rows - 1) * 2) + 2}'

    def _serialize_datetime(value):
        return pd.Timestamp(value).isoformat()

    def _xaxis_range_update(start_date, end_date):
        start_serialized = _serialize_datetime(start_date)
        end_serialized = _serialize_datetime(end_date)
        return {
            f'{root_left_axis}.range[0]': start_serialized,
            f'{root_left_axis}.range[1]': end_serialized,
            f'{root_right_axis}.range[0]': start_serialized,
            f'{root_right_axis}.range[1]': end_serialized,
        }

    zoom_buttons = []
    for label, offset in zoom_ranges:
        start_date = max(min_date, end_date - offset)
        zoom_buttons.append(
            dict(
                label=label,
                method='relayout',
                args=[_xaxis_range_update(start_date, end_date)]
            )
        )

    subplot_titles = []
    for column in plot_columns:
        display_name = display_name_map.get(column, column) if display_name_map else column
        subplot_titles.append(f'{display_name} - Sharpe')
        subplot_titles.append(f'{display_name} - Spread')

    asset_palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    asset_colors = {
        column: asset_palette[idx % len(asset_palette)]
        for idx, column in enumerate(plot_columns)
    }

    subplot_spacing = min(0.05, 1.0 / max(num_rows * 4, 8)) if num_rows > 1 else 0.0
    row_height = 300 if left_style or right_style else 240
    fig = make_subplots(
        rows=num_rows,
        cols=2,
        shared_xaxes='columns',
        vertical_spacing=subplot_spacing,
        horizontal_spacing=0.08,
        subplot_titles=subplot_titles
    )
    for annotation in fig.layout.annotations:
        annotation.font = dict(size=10, color='rgba(220, 220, 220, 0.90)')

    for row_idx, column in enumerate(plot_columns, start=1):
        display_name = display_name_map.get(column, column) if display_name_map else column
        if column in left_columns:
            fig.add_trace(
                go.Scatter(
                    x=left_plot_data.index,
                    y=left_plot_data[column],
                    mode='lines',
                    name=f'{display_name} - Sharpe',
                    line=dict(color=asset_colors[column], width=2),
                    showlegend=False
                ),
                row=row_idx,
                col=1
            )

            if left_style:
                _apply_reference_panel_style(fig, row_idx, 1, left_style, add_zero_line=left_add_zero_line, sigma_lines=left_sigma_lines)
            else:
                if left_add_zero_line:
                    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=row_idx, col=1)
                if left_sigma_lines:
                    for sigma in left_sigma_lines:
                        fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=1)
                        fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=1)
            fig.update_yaxes(title_text=left_yaxis_title, row=row_idx, col=1)

        if column in right_columns:
            fig.add_trace(
                go.Scatter(
                    x=right_plot_data.index,
                    y=right_plot_data[column],
                    mode='lines',
                    name=f'{display_name} - Spread',
                    line=dict(color=asset_colors[column], width=2),
                    showlegend=False
                ),
                row=row_idx,
                col=2
            )

            if right_style:
                _apply_reference_panel_style(fig, row_idx, 2, right_style, add_zero_line=right_add_zero_line, sigma_lines=right_sigma_lines)
            else:
                if right_add_zero_line:
                    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=row_idx, col=2)
                if right_sigma_lines:
                    for sigma in right_sigma_lines:
                        fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=2)
                        fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=2)
            fig.update_yaxes(title_text=right_yaxis_title, row=row_idx, col=2)

    fig.update_layout(
        title=title,
        height=max(620, row_height * num_rows + 180),
        template='plotly_dark',
        autosize=True,
        margin=dict(l=40, r=40, t=90, b=40),
        updatemenus=[
            dict(
                buttons=zoom_buttons,
                direction='down',
                showactive=True,
                active=4,
                x=0.01,
                y=1.12,
                xanchor='left',
                yanchor='top'
            )
        ]
    )

    fig.update_xaxes(range=[default_start, end_date], row=num_rows, col=1)
    fig.update_xaxes(range=[default_start, end_date], row=num_rows, col=2)
    fig.update_xaxes(title_text='Date', row=num_rows, col=1)
    fig.update_xaxes(title_text='Date', row=num_rows, col=2)
    fig.show(config={'responsive': True})

if broad_market_close.empty:
    print('No broad market reference data available.')
else:
    broad_market_close_plot = broad_market_close.ffill()

    broad_risk_free_daily_rate = pd.Series(risk_free_rate, index=broad_market_close.index, dtype=float)
    broad_risk_free_proxy = _normalize_market_index(yf.Ticker('^IRX').history(period=broad_market_reference_period, interval=interval))
    if not broad_risk_free_proxy.empty and 'Close' in broad_risk_free_proxy:
        broad_risk_free_annual_yield = broad_risk_free_proxy['Close'].dropna().sort_index().div(100)
        if not broad_risk_free_annual_yield.empty:
            broad_risk_free_daily_rate = ((1 + broad_risk_free_annual_yield) ** (1 / 252) - 1).shift(1)
            broad_risk_free_daily_rate = broad_risk_free_daily_rate.reindex(broad_market_close.index).ffill()
            if broad_risk_free_daily_rate.dropna().empty:
                broad_risk_free_daily_rate = pd.Series(risk_free_rate, index=broad_market_close.index, dtype=float)

    broad_market_reference_benchmark = 'U.S. Large Cap'
    if broad_market_reference_benchmark not in broad_market_close_map:
        broad_market_reference_benchmark = next(iter(broad_market_close_map))

    benchmark_close_reference = broad_market_close_map[broad_market_reference_benchmark].dropna().sort_index()
    broad_market_rolling_sharpe_series = {}
    broad_market_sharpe_spread_series = {}
    for asset_name, asset_close in broad_market_close_map.items():
        analysis_index = asset_close.index.intersection(benchmark_close_reference.index).sort_values()
        if analysis_index.empty:
            continue

        asset_close_aligned = asset_close.reindex(analysis_index).dropna()
        benchmark_close_aligned = benchmark_close_reference.reindex(analysis_index).dropna()
        analysis_index = asset_close_aligned.index.intersection(benchmark_close_aligned.index).sort_values()
        if analysis_index.empty:
            continue

        asset_close_aligned = asset_close_aligned.reindex(analysis_index)
        benchmark_close_aligned = benchmark_close_aligned.reindex(analysis_index)
        aligned_risk_free_daily_rate = broad_risk_free_daily_rate.reindex(analysis_index).ffill()
        if aligned_risk_free_daily_rate.dropna().empty:
            aligned_risk_free_daily_rate = pd.Series(risk_free_rate, index=analysis_index, dtype=float)

        asset_sharpe = rolling.risk_adjusted_returns(
            asset_close_aligned,
            windows=[time_frame_long],
            ratio_type='sharpe',
            risk_free_rate=aligned_risk_free_daily_rate,
        ).iloc[:, 0]
        benchmark_sharpe = rolling.risk_adjusted_returns(
            benchmark_close_aligned,
            windows=[time_frame_long],
            ratio_type='sharpe',
            risk_free_rate=aligned_risk_free_daily_rate,
        ).iloc[:, 0].reindex(asset_sharpe.index)

        broad_market_rolling_sharpe_series[asset_name] = asset_sharpe
        broad_market_sharpe_spread_series[asset_name] = benchmark_sharpe - asset_sharpe

    broad_market_rolling_sharpe = pd.DataFrame(broad_market_rolling_sharpe_series).dropna(how='all')
    broad_market_rolling_sharpe_zscore = broad_market_rolling_sharpe.apply(calculate_zscore).dropna(how='all')

    broad_market_sharpe_spread = pd.DataFrame(broad_market_sharpe_spread_series).dropna(how='all')
    broad_market_sharpe_spread_zscore = broad_market_sharpe_spread.apply(calculate_zscore).dropna(how='all')
    broad_market_sharpe_spread_zscore = broad_market_sharpe_spread_zscore.drop(
        columns=[broad_market_reference_benchmark],
        errors='ignore'
    )

    broad_market_reference_benchmark_display = broad_market_display_name_map.get(
        broad_market_reference_benchmark,
        broad_market_reference_benchmark
    )
    broad_market_benchmark_sharpe_zscore = broad_market_rolling_sharpe_zscore[[broad_market_reference_benchmark]].dropna(how='all')
    broad_market_relative_sharpe_zscore = broad_market_rolling_sharpe_zscore.drop(
        columns=[broad_market_reference_benchmark],
        errors='ignore'
    )

    plot_cumulative_return_overlay(
        broad_market_close_plot,
        'Broad Market Reference - Cumulative Returns',
        display_name_map=broad_market_display_name_map
    )
    plot_reference_dataframe(
        broad_market_benchmark_sharpe_zscore,
        f'Broad Market Reference - {broad_market_reference_benchmark_display} Rolling Sharpe Z-Score',
        'Rolling Sharpe Z-Score',
        add_zero_line=True,
        sigma_lines=[1, 2, 3],
        display_name_map=broad_market_display_name_map,
        style='risk_detail'
    )
    plot_reference_dataframes_side_by_side(
        broad_market_relative_sharpe_zscore,
        broad_market_sharpe_spread_zscore,
        f'Broad Market Reference - Rolling Sharpe and Spread Z-Scores vs {broad_market_reference_benchmark_display}',
        'Rolling Sharpe Z-Score',
        'Sharpe Spread Z-Score',
        left_add_zero_line=True,
        right_add_zero_line=True,
        left_sigma_lines=[1, 2, 3],
        right_sigma_lines=[0.5, 1, 1.5, 2],
        left_style='risk_detail',
        right_style='risk_summary',
        display_name_map=broad_market_display_name_map
    )


In [ ]:
#5 Load: retrieve all tickers / prices 
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Load: retrieve all tickers / prices / spreads for major markets
sp500 = yf.download('SPY', period=period, interval=interval,progress=False)
risk_free_rate = yf.download('^IRX', period=period, interval=interval, progress=False)
#market_assets = qe.get_market_assets()

market_assets = market_data.get_market_assets()

if asset_class == 'broad':
    #Load: retrieve prices
    indices_df             = market_data.generate_series(market_assets['INDICES'], columns=['Close'],period=period, interval=interval)
    sectors_df            = market_data.generate_series(market_assets['SECTORS'], columns=['Close'],period=period, interval=interval)
    industries_df         = market_data.generate_series(market_assets['INDUSTRIES'], columns=['Close'],period=period, interval=interval)
    bonds_df = market_data.generate_series(market_assets['BONDS'], columns=['Close'],period=period, interval=interval)
    precious_metals_df = market_data.generate_series(market_assets['PRECIOUS_METALS'], columns=['Close'],period=period, interval=interval)
    crypto_df = market_data.generate_series(market_assets['CRYPTO'], columns=['Close'],period=period, interval=interval)
    #crypto_df = crypto_df.loc[sp500.index]
    energy_df = market_data.generate_series(market_assets['ENERGY'], columns=['Close'],period=period, interval=interval)
    foreign_markets_df = market_data.generate_series(market_assets['FOREIGN_MARKETS'], columns=['Close'],period=period, interval=interval)
    primary_sector_etfs_df = market_data.generate_series(market_assets['PRIMARY_SECTORS'], columns=['Close'],period=period, interval=interval)
    capitalizations_df = market_data.generate_series(market_assets['CAPITALIZATIONS'], columns=['Close'],period=period, interval=interval)
    innovation_df = market_data.generate_series(market_assets['INNOVATION'], columns=['Close'],period=period, interval=interval)
    long_leveraged_df = market_data.generate_series(market_assets['LONG_LEVERAGE'], columns=['Close'],period=period, interval=interval)
    short_leveraged_df = market_data.generate_series(market_assets['SHORT_LEVERAGE'], columns=['Close'],period=period, interval=interval)
    single_factor_df = market_data.generate_series(market_assets['SINGLE_FACTOR'], columns=['Close'],period=period, interval=interval)
    multi_factor_df = market_data.generate_series(market_assets['MULTI_FACTOR'], columns=['Close'],period=period, interval=interval)
    minimum_volatility_df = market_data.generate_series(market_assets['MINIMUM_VOLATILITY'], columns=['Close'],period=period, interval=interval)


    etf_prices = pd.concat([
        indices_df,
        sectors_df,
        industries_df,
        bonds_df,
        precious_metals_df,
        #crypto_df,
        energy_df,
        foreign_markets_df,
        primary_sector_etfs_df,
        capitalizations_df,
        innovation_df,
        long_leveraged_df,
        short_leveraged_df,
        single_factor_df,
        multi_factor_df,
        minimum_volatility_df
    ], axis=1).loc[:, lambda df: ~df.columns.duplicated()]

    #test
    test_prices = pd.concat([indices_df,
                            sectors_df,
                            crypto_df
                            ], axis=1).loc[:, lambda df: ~df.columns.duplicated()]

    benchmark_series           = etf_prices[benchmark]

    # List of dataframes for week and short time frames
    etf_dataframes = {
        "indices": indices_df, 
        "sectors": sectors_df, 
        "industries": industries_df, 
        "bonds": bonds_df, 
        "precious_metals": precious_metals_df, 
        #"crypto": crypto_df, 
        "energy": energy_df,
        "foreign_markets": foreign_markets_df, 
        "primary_sector_etfs": primary_sector_etfs_df, 
        "capitalizations": capitalizations_df, 
        "innovation": innovation_df,
        "long_leveraged": long_leveraged_df,
        "short_leveraged": short_leveraged_df,
        "single_factor": single_factor_df,
        "multi_factor": multi_factor_df,
        "minimum_volatility": minimum_volatility_df
    }
 
    print('Computing the correlation matrix...')
    etf_dataframes_correlation_matrices = {key: value.corr() for key, value in etf_dataframes.items()}
elif asset_class == 'equity':
    xlk_holdings_df = market_data.generate_series(market_assets['XLK_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlf_holdings_df = market_data.generate_series(market_assets['XLF_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xli_holdings_df = market_data.generate_series(market_assets['XLI_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlb_holdings_df = market_data.generate_series(market_assets['XLB_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlv_holdings_df = market_data.generate_series(market_assets['XLV_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlu_holdings_df = market_data.generate_series(market_assets['XLU_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xly_holdings_df = market_data.generate_series(market_assets['XLY_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlc_holdings_df = market_data.generate_series(market_assets['XLC_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlp_holdings_df = market_data.generate_series(market_assets['XLP_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xle_holdings_df = market_data.generate_series(market_assets['XLE_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlre_holdings_df = market_data.generate_series(market_assets['XLRE_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    indices_df = market_data.generate_series(market_assets['INDICES'], columns=['Close'], period=period, interval=interval)
    etf_prices = pd.concat([indices_df], axis=1).loc[:, lambda df: ~df.columns.duplicated()]
    benchmark_series = etf_prices[benchmark]
    etf_dataframes = {
        "indices": indices_df,
        "xlk_holdings": xlk_holdings_df,
        "xlf_holdings": xlf_holdings_df,
        "xli_holdings": xli_holdings_df,
        "xlb_holdings": xlb_holdings_df,
        "xlv_holdings": xlv_holdings_df,
        "xlu_holdings": xlu_holdings_df,
        "xly_holdings": xly_holdings_df,
        "xlc_holdings": xlc_holdings_df,
        "xlp_holdings": xlp_holdings_df,
        "xle_holdings": xle_holdings_df,
        "xlre_holdings": xlre_holdings_df
    }
    # Mapping of sectors to their respective ETF holdings dataframes
    sector_mapping = {
        'energy': ('xle_holdings', xle_holdings_df),
        'materials': ('xlb_holdings', xlb_holdings_df),
        'industrials': ('xli_holdings', xli_holdings_df),
        'consumer_discretionary': ('xly_holdings', xly_holdings_df),
        'consumer_staples': ('xlp_holdings', xlp_holdings_df),
        'healthcare': ('xlv_holdings', xlv_holdings_df),
        'financials': ('xlf_holdings', xlf_holdings_df),
        'information_technology': ('xlk_holdings', xlk_holdings_df),
        'communication_services': ('xlc_holdings', xlc_holdings_df),
        'utilities': ('xlu_holdings', xlu_holdings_df),
        'real_estate': ('xlre_holdings', xlre_holdings_df)
    }
    
    if sector == 'all':
        # Include all sectors by updating the etf_dataframes dictionary
        etf_dataframes.update({key: df for key, df in sector_mapping.values()})
    elif sector in sector_mapping:
        # Include only the specific sector
        key, df = sector_mapping[sector]
        etf_dataframes = {
            "indices": indices_df,
            key: df
        }
        etf_prices = pd.concat([indices_df, df], axis=1).loc[:, lambda df: ~df.columns.duplicated()]
    else:
        supported_values = "'all', " + ", ".join([f"'{s}'" for s in sector_mapping.keys()])
        raise ValueError(f"Unsupported sector. Supported values are {supported_values}.")
    etf_dataframes_correlation_matrices = {key: value.corr() for key, value in etf_dataframes.items()}    
    
else:
    raise ValueError("Unsupported asset class. Supported values are 'broad' and 'equity'.")


agg_dataframe = pd.concat(list(etf_dataframes.values()), axis=1)
agg_dataframe = agg_dataframe.loc[:,~agg_dataframe.columns.duplicated()]

In [ ]:
#6 Computations...

#1. return spreads between benchmark and all assets
#2. the sortino ratios for all assets
#3. the spreads between the sortino ratios of all assets and the benchmark


#compute the weekly, short, mid, and long term returns for the benchmark
sp500_monthly_returns = series_transforms.returns(sp500,frequency='monthly')
sp500_weekly_returns  = series_transforms.returns(sp500,frequency='weekly')
sp500_daily_returns   = series_transforms.returns(sp500,frequency='daily')

#the spreads between the benchmark and all assets
#-----------------------------------------------------------
#the spreads between the benchmark and all assets
#Create spreads for weekly time frame
print("Computing spreads between benchmark and all assets...")
benchmark_minus_etf_week = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=time_frame_week, mode=mode)

#Create spreads for 3 day time frame
print("Computing spreads between benchmark and all assets for the 3 day time frame...")
benchmark_minus_etf_3 = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=3, mode=mode)

#Create spreads for 9 day time frame
print("Computing spreads between benchmark and all assets for the 9 day time frame...")
benchmark_minus_etf_9 = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=9, mode=mode)

#display(benchmark_minus_etf_week)
#Create spreads for short time frame
print(f"Computing spreads between benchmark and all assets for the short time frame ({time_frame_short} days)...")
benchmark_minus_etf_short = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=time_frame_short, mode=mode)

#Create spreads for mid time frame
print(f"Computing spreads between benchmark and all assets for the mid time frame ({time_frame_mid} days)...")
benchmark_minus_etf_mid = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=time_frame_mid, mode=mode)

#create spreads for long time frame
print(f"Computing spreads between benchmark and all assets for the long time frame ({time_frame_long} days)...")
benchmark_minus_etf_long = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=time_frame_long, mode=mode)

#Create spreads for 400 day time frame
print("Computing spreads between benchmark and all assets for the 400 day time frame...")
benchmark_minus_etf_400 = risk_relative_analytics.spread(etf_prices, benchmark_series, time_frame=400, mode=mode)

print(" ")
print("-----------------------------------------------------------------------")

#-----------------------------------------------------------

#the sortino ratios for all assets
print("computing the rolling sortino ratios for all assets for the 3 day time frame...")
rolling_sortino_ratios_etf_3 = rolling.risk_adjusted_returns(etf_prices, windows=[3], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print("computing the rolling sortino ratios for all assets for the 9 day time frame...")
rolling_sortino_ratios_etf_9 = rolling.risk_adjusted_returns(etf_prices, windows=[9], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(f"computing the rolling sortino ratios for all assets for the short time frame ({time_frame_short} days)...")
rolling_sortino_ratios_etf_21 = rolling.risk_adjusted_returns(etf_prices, windows=[21], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(f"computing the rolling sortino ratios for all assets for the mid time frame ({time_frame_mid} days)...")
rolling_sortino_ratios_etf_50 = rolling.risk_adjusted_returns(etf_prices, windows=[50], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(f"computing the rolling sortino ratios for all assets for the long time frame ({time_frame_long} days)...")
rolling_sortino_ratios_etf_200 = rolling.risk_adjusted_returns(etf_prices, windows=[200], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print("computing the rolling sortino ratios for all assets for the 400 day time frame...")
rolling_sortino_ratios_etf_400 = rolling.risk_adjusted_returns(etf_prices, windows=[400], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(" ")
print("-----------------------------------------------------------------------")

#the spreads between the sortino ratios of all assets and the benchmark
print("computing the rolling sortino ratios for all assets minus the benchmark for the 3 day time frame...")
rolling_sortino_ratios_benchmark_minus_etf_3  = -rolling_sortino_ratios_etf_3.sub(rolling_sortino_ratios_etf_3['SPY'], axis=0)

print("computing the rolling sortino ratios for all assets minus the benchmark for the 9 day time frame...")
rolling_sortino_ratios_benchmark_minus_etf_9  = -rolling_sortino_ratios_etf_9.sub(rolling_sortino_ratios_etf_9['SPY'], axis=0)

print(f"computing the rolling sortino ratios for all assets minus the benchmark for the short time frame ({time_frame_short} days)...")
rolling_sortino_ratios_benchmark_minus_etf_21  = -rolling_sortino_ratios_etf_21.sub(rolling_sortino_ratios_etf_21['SPY'], axis=0)

print(f"computing the rolling sortino ratios for all assets minus the benchmark for the mid time frame ({time_frame_mid} days)...")
rolling_sortino_ratios_benchmark_minus_etf_50  = -rolling_sortino_ratios_etf_50.sub(rolling_sortino_ratios_etf_50['SPY'], axis=0)

print(f"computing the rolling sortino ratios for all assets minus the benchmark for the long time frame ({time_frame_long} days)...")
rolling_sortino_ratios_benchmark_minus_etf_200  = -rolling_sortino_ratios_etf_200.sub(rolling_sortino_ratios_etf_200['SPY'], axis=0)

print("computing the rolling sortino ratios for all assets minus the benchmark for the 400 day time frame...")
rolling_sortino_ratios_benchmark_minus_etf_400  = -rolling_sortino_ratios_etf_400.sub(rolling_sortino_ratios_etf_400['SPY'], axis=0)

print(f"computing 10 year correlation metrics to {benchmark}...")
etf_daily_returns = etf_prices.pct_change(fill_method=None)
correlation_end = etf_daily_returns.index.max()
correlation_start = correlation_end - pd.DateOffset(years=10)
correlation_window = etf_daily_returns.loc[correlation_start:correlation_end]
correlation_tolerance = pd.Timedelta(days=45)
minimum_downside_observations = 30
correlation_metrics_10y = {
    'Pearson': {},
    'Spearman': {},
    'Kendall': {},
    'Downside Pearson': {}
}

for ticker in correlation_window.columns:
    aligned_returns = pd.concat([
        correlation_window[benchmark],
        correlation_window[ticker]
    ], axis=1).dropna()

    has_full_lookback = (
        not aligned_returns.empty
        and aligned_returns.index.min() <= correlation_start + correlation_tolerance
    )

    if has_full_lookback:
        benchmark_returns = aligned_returns.iloc[:, 0]
        asset_returns = aligned_returns.iloc[:, 1]
        downside_mask = benchmark_returns < 0
        downside_returns = aligned_returns.loc[downside_mask]

        correlation_metrics_10y['Pearson'][ticker] = benchmark_returns.corr(asset_returns, method='pearson')
        correlation_metrics_10y['Spearman'][ticker] = benchmark_returns.corr(asset_returns, method='spearman')
        correlation_metrics_10y['Kendall'][ticker] = benchmark_returns.corr(asset_returns, method='kendall')
        correlation_metrics_10y['Downside Pearson'][ticker] = (
            downside_returns.iloc[:, 0].corr(downside_returns.iloc[:, 1], method='pearson')
            if len(downside_returns) >= minimum_downside_observations else np.nan
        )
    else:
        for metric_name in correlation_metrics_10y:
            correlation_metrics_10y[metric_name][ticker] = np.nan

correlation_metrics_10y = {
    metric_name: pd.Series(metric_values)
    for metric_name, metric_values in correlation_metrics_10y.items()
}
correlation_10y = correlation_metrics_10y['Pearson']
'''
print(" ")
print("-----------------------------------------------------------------------")

print("Computing pairwise spreads between assets within each category...")
print(f"Computing pairwise spreads for the short time frame ({time_frame_short} days)...")
pairwise_spreads_21 = rolling.create_pairwise_spreads(etf_dataframes, window=time_frame_short)

print(f"Computing pairwise spreads for the mid time frame ({time_frame_mid} days)...")
pairwise_spreads_50 = rolling.create_pairwise_spreads(etf_dataframes, window=time_frame_mid)

print(f"Computing pairwise spreads for the long time frame ({time_frame_long} days)...")
pairwise_spreads_200 = rolling.create_pairwise_spreads(etf_dataframes, window=time_frame_long)

'''

In [ ]:
#7 Correlation Comparison by Metric
correlation_palette = {
    'Q1 (Lowest)': '#ef4444',
    'Q2': '#f59e0b',
    'Q3': '#60a5fa',
    'Q4 (Highest)': '#22c55e'
}
correlation_plot_frames = {}
pearson_correlation_series = correlation_metrics_10y.get('Pearson', pd.Series(dtype=float))

for metric_name, metric_series in correlation_metrics_10y.items():
    metric_column = f'10 year {metric_name} Correlation to {benchmark}'
    metric_plot_df = metric_series.rename(metric_column).dropna().rename_axis('Ticker').reset_index()
    if metric_plot_df.empty:
        continue

    pearson_rank_reference = pearson_correlation_series.reindex(metric_plot_df['Ticker']).dropna().sort_values()
    pearson_rank_map = pd.Series(
        np.arange(1, len(pearson_rank_reference) + 1),
        index=pearson_rank_reference.index
    )

    quantile_count = min(4, len(metric_plot_df))
    quantile_labels = ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'][:quantile_count]
    metric_plot_df['Quantile'] = pd.qcut(
        metric_plot_df[metric_column].rank(method='first'),
        q=quantile_count,
        labels=quantile_labels
    )
    metric_plot_df = metric_plot_df.sort_values(by=metric_column, ascending=True).reset_index(drop=True)
    metric_plot_df['Position'] = np.arange(len(metric_plot_df))
    metric_plot_df['Current Rank'] = np.arange(1, len(metric_plot_df) + 1)
    metric_plot_df['Pearson Rank'] = metric_plot_df['Ticker'].map(pearson_rank_map)
    metric_plot_df['Rank Shift vs Pearson'] = metric_plot_df['Pearson Rank'] - metric_plot_df['Current Rank']
    metric_plot_df['Metric'] = metric_name
    metric_plot_df['Current Rank Label'] = metric_plot_df['Current Rank'].astype(int).astype(str)
    metric_plot_df['Pearson Rank Label'] = metric_plot_df['Pearson Rank'].apply(lambda value: 'N/A' if pd.isna(value) else str(int(value)))
    metric_plot_df['Rank Shift Label'] = metric_plot_df['Rank Shift vs Pearson'].apply(
        lambda value: 'No change'
        if pd.isna(value) or value == 0
        else f'Up {int(abs(value))} positions' if value > 0
        else f'Down {int(abs(value))} positions'
    )
    metric_plot_df['Color'] = metric_plot_df['Quantile'].map(correlation_palette)
    metric_plot_df['Shift Color'] = metric_plot_df['Rank Shift vs Pearson'].apply(
        lambda value: '#22c55e' if value > 0 else '#ef4444' if value < 0 else '#9ca3af'
    )
    correlation_plot_frames[metric_name] = (metric_column, metric_plot_df)

if correlation_plot_frames:
    default_metric = 'Pearson' if 'Pearson' in correlation_plot_frames else next(iter(correlation_plot_frames))
    default_column, default_plot_df = correlation_plot_frames[default_metric]
    default_positions = default_plot_df['Position'].tolist()
    default_ticker_order = default_plot_df['Ticker'].tolist()
    default_average_correlation = default_plot_df[default_column].mean()

    correlation_fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=('Selected Correlation Metric', 'Rank Shift vs Pearson Order')
    )
    correlation_fig.add_trace(
        go.Bar(
            x=default_plot_df['Position'],
            y=default_plot_df[default_column],
            marker_color=default_plot_df['Color'],
            customdata=np.array(default_plot_df[['Ticker', 'Metric', 'Quantile', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']]),
            hovertemplate='Ticker: %{customdata[0]}<br>Metric: %{customdata[1]}<br>Value: %{y:.2f}<br>Quantile: %{customdata[2]}<br>Current Rank: %{customdata[3]}<br>Pearson Rank: %{customdata[4]}<br>Shift vs Pearson: %{customdata[5]}<extra></extra>'
        ),
        row=1,
        col=1
    )
    correlation_fig.add_trace(
        go.Scatter(
            x=default_plot_df['Position'],
            y=np.full(len(default_plot_df), default_average_correlation, dtype=float),
            mode='lines',
            name='Average Correlation',
            line=dict(color='rgba(255, 255, 255, 0.90)', width=2, dash='dash'),
            hovertemplate='Average Correlation: %{y:.2f}<extra></extra>',
            showlegend=True
        ),
        row=1,
        col=1
    )
    correlation_fig.add_trace(
        go.Bar(
            x=default_plot_df['Position'],
            y=default_plot_df['Rank Shift vs Pearson'],
            marker_color=default_plot_df['Shift Color'],
            customdata=np.array(default_plot_df[['Ticker', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']]),
            hovertemplate='Ticker: %{customdata[0]}<br>Rank Shift: %{y:+.0f}<br>Current Rank: %{customdata[1]}<br>Pearson Rank: %{customdata[2]}<br>Interpretation: %{customdata[3]}<extra></extra>'
        ),
        row=2,
        col=1
    )
    correlation_fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)

    metric_buttons = []
    for metric_name, (metric_column, metric_plot_df) in correlation_plot_frames.items():
        metric_average_correlation = metric_plot_df[metric_column].mean()
        metric_buttons.append(
            dict(
                label=metric_name,
                method='update',
                args=[
                    {
                        'x': [metric_plot_df['Position'], metric_plot_df['Position'], metric_plot_df['Position']],
                        'y': [
                            metric_plot_df[metric_column],
                            np.full(len(metric_plot_df), metric_average_correlation, dtype=float),
                            metric_plot_df['Rank Shift vs Pearson']
                        ],
                        'marker.color': [metric_plot_df['Color'], None, metric_plot_df['Shift Color']],
                        'customdata': [
                            np.array(metric_plot_df[['Ticker', 'Metric', 'Quantile', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']]),
                            None,
                            np.array(metric_plot_df[['Ticker', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']])
                        ]
                    },
                    {
                        'title': f'10 Year {metric_name} Correlation to {benchmark} by Quantile',
                        'xaxis.tickmode': 'array',
                        'xaxis.tickvals': metric_plot_df['Position'].tolist(),
                        'xaxis.ticktext': metric_plot_df['Ticker'].tolist(),
                        'xaxis2.tickmode': 'array',
                        'xaxis2.tickvals': metric_plot_df['Position'].tolist(),
                        'xaxis2.ticktext': metric_plot_df['Ticker'].tolist(),
                        'yaxis.title.text': 'Correlation',
                        'yaxis2.title.text': 'Rank Shift'
                    }
                ]
            )
        )

    correlation_fig.update_layout(
        title=f'10 Year {default_metric} Correlation to {benchmark} by Quantile',
        template='plotly_dark',
        autosize=True,
        height=760,
        margin=dict(l=40, r=40, t=120, b=120),
        updatemenus=[
            dict(
                buttons=metric_buttons,
                direction='down',
                showactive=True,
                x=0.01,
                y=1.22,
                xanchor='left',
                yanchor='top'
            )
        ],
        annotations=[
            dict(
                text='Quantiles: Q1 lowest, Q4 highest',
                xref='paper',
                yref='paper',
                x=1,
                y=1.22,
                showarrow=False,
                xanchor='right',
                yanchor='top',
                font=dict(size=11, color='white')
            )
        ]
    )
    correlation_fig.update_xaxes(title='', tickmode='array', tickvals=default_positions, ticktext=default_ticker_order, showticklabels=False, row=1, col=1)
    correlation_fig.update_xaxes(title='', tickmode='array', tickvals=default_positions, ticktext=default_ticker_order, tickangle=45, row=2, col=1)
    correlation_fig.update_yaxes(title='Correlation', row=1, col=1)
    correlation_fig.update_yaxes(title='Rank Shift', row=2, col=1)
    correlation_fig.show(config={'responsive': True})
else:
    print(f'No valid 10 year correlation data available for {benchmark}.')


In [ ]:
#7A Average Rolling Correlation Over Time
rolling_correlation_windows = {
    '21 Day Average Rolling Correlation': 21,
    '50 Day Average Rolling Correlation': 50,
    '200 Day Average Rolling Correlation': 200
}

rolling_correlation_universe = correlation_10y.dropna().index.tolist() if isinstance(correlation_10y, pd.Series) else []
if benchmark in rolling_correlation_universe and len(rolling_correlation_universe) > 1:
    rolling_correlation_universe = [ticker for ticker in rolling_correlation_universe if ticker != benchmark]
rolling_correlation_universe = [
    ticker for ticker in rolling_correlation_universe
    if ticker in etf_daily_returns.columns and ticker != benchmark
]

if benchmark not in etf_daily_returns.columns:
    print(f'Benchmark {benchmark} is not available in the daily return frame.')
elif not rolling_correlation_universe:
    print(f'No valid correlation universe available to compute average rolling correlation to {benchmark}.')
else:
    benchmark_daily_returns = etf_daily_returns[benchmark]
    rolling_average_correlation_map = {}

    for label, window in rolling_correlation_windows.items():
        rolling_correlation_frame = etf_daily_returns[rolling_correlation_universe].apply(
            lambda asset_returns: asset_returns.rolling(window).corr(benchmark_daily_returns)
        )
        rolling_average_correlation_map[label] = rolling_correlation_frame.mean(axis=1, skipna=True).dropna()

    non_empty_rolling_average_map = {
        label: series
        for label, series in rolling_average_correlation_map.items()
        if not series.empty
    }

    if not non_empty_rolling_average_map:
        print(f'No rolling average correlation data available for {benchmark}.')
    else:
        rolling_correlation_colors = {
            '21 Day Average Rolling Correlation': '#38bdf8',
            '50 Day Average Rolling Correlation': '#f59e0b',
            '200 Day Average Rolling Correlation': '#22c55e'
        }

        rolling_average_correlation_fig = make_subplots(
            rows=len(non_empty_rolling_average_map),
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.06,
            subplot_titles=list(non_empty_rolling_average_map.keys())
        )

        for annotation in rolling_average_correlation_fig.layout.annotations:
            annotation.font = dict(size=11, color='rgba(220, 220, 220, 0.90)')

        for row_idx, (label, correlation_series) in enumerate(non_empty_rolling_average_map.items(), start=1):
            correlation_mean = correlation_series.mean()
            correlation_min = min(correlation_series.min(), 0, correlation_mean)
            correlation_max = max(correlation_series.max(), 0, correlation_mean)
            correlation_span = correlation_max - correlation_min
            correlation_padding = max(correlation_span * 0.08, 0.02) if pd.notna(correlation_span) else 0.02
            rolling_average_correlation_fig.add_trace(
                go.Scatter(
                    x=correlation_series.index,
                    y=correlation_series,
                    mode='lines',
                    name=label,
                    line=dict(color=rolling_correlation_colors.get(label, '#60a5fa'), width=2),
                    hovertemplate='Date: %{x|%Y-%m-%d}<br>Average Correlation: %{y:.2f}<extra></extra>',
                    showlegend=False
                ),
                row=row_idx,
                col=1
            )
            rolling_average_correlation_fig.add_hline(
                y=correlation_mean,
                line_dash='dash',
                line_color='rgba(235, 235, 235, 0.75)',
                row=row_idx,
                col=1
            )
            rolling_average_correlation_fig.add_annotation(
                x=0.985,
                y=correlation_mean,
                xref='x domain',
                yref='y',
                text=f'Mean {correlation_mean:.2f}',
                showarrow=False,
                xanchor='right',
                yanchor='bottom',
                font=dict(size=10, color='rgba(235, 235, 235, 0.90)'),
                row=row_idx,
                col=1
            )
            rolling_average_correlation_fig.add_hline(
                y=0,
                line_dash='dot',
                line_color='rgba(180, 180, 180, 0.65)',
                row=row_idx,
                col=1
            )
            rolling_average_correlation_fig.add_annotation(
                x=0.985,
                y=0,
                xref='x domain',
                yref='y',
                text='Zero',
                showarrow=False,
                xanchor='right',
                yanchor='top',
                font=dict(size=10, color='rgba(200, 200, 200, 0.85)'),
                row=row_idx,
                col=1
            )
            rolling_average_correlation_fig.update_yaxes(
                title_text='Avg Corr',
                range=[correlation_min - correlation_padding, correlation_max + correlation_padding],
                row=row_idx,
                col=1
            )

        rolling_average_correlation_fig.update_layout(
            title=f'Average Rolling Correlation to {benchmark} Over Time',
            template='plotly_dark',
            autosize=True,
            height=max(780, 260 * len(non_empty_rolling_average_map) + 120),
            margin=dict(l=50, r=40, t=90, b=40)
        )
        rolling_average_correlation_fig.update_xaxes(title_text='Date', row=len(non_empty_rolling_average_map), col=1)
        rolling_average_correlation_fig.show(config={'responsive': True})


In [ ]:
#8 Create DataFrames for each time frame
z_score_3 = pd.DataFrame()
z_score_9 = pd.DataFrame()
z_score_21 = pd.DataFrame()
z_score_50 = pd.DataFrame()
z_score_200 = pd.DataFrame()
z_score_400 = pd.DataFrame()

# Calculate z-scores for 3-day time frame
z_score_sortino_ratio_3 = signal_labels.z_score(rolling_sortino_ratios_etf_3)
z_score_benchmark_minus_etf_3 = signal_labels.z_score(rolling_sortino_ratios_benchmark_minus_etf_3)

z_score_3['3 day Sortino Ratio (z score)'] = z_score_sortino_ratio_3
z_score_3['3 day Benchmark Minus ETF Sortino Ratio (z score)'] = z_score_benchmark_minus_etf_3
z_score_3.sort_values(by='3 day Sortino Ratio (z score)', ascending=True, inplace=True)

# Calculate z-scores for 9-day time frame
z_score_sortino_ratio_9 = signal_labels.z_score(rolling_sortino_ratios_etf_9)
z_score_benchmark_minus_etf_9 = signal_labels.z_score(rolling_sortino_ratios_benchmark_minus_etf_9)

z_score_9['9 day Sortino Ratio (z score)'] = z_score_sortino_ratio_9
z_score_9['9 day Benchmark Minus ETF Sortino Ratio (z score)'] = z_score_benchmark_minus_etf_9
z_score_9.sort_values(by='9 day Sortino Ratio (z score)', ascending=True, inplace=True)

# Calculate z-scores for 21-day time frame
z_score_sortino_ratio_21 = signal_labels.z_score(rolling_sortino_ratios_etf_21)
z_score_benchmark_minus_etf_21 = signal_labels.z_score(rolling_sortino_ratios_benchmark_minus_etf_21)

z_score_21['21 day Sortino Ratio (z score)'] = z_score_sortino_ratio_21
z_score_21['21 day Benchmark Minus ETF Sortino Ratio (z score)'] = z_score_benchmark_minus_etf_21
z_score_21.sort_values(by='21 day Sortino Ratio (z score)', ascending=True, inplace=True)

# Calculate z-scores for 50-day time frame
z_score_sortino_ratio_50 = signal_labels.z_score(rolling_sortino_ratios_etf_50)
z_score_benchmark_minus_etf_50 = signal_labels.z_score(rolling_sortino_ratios_benchmark_minus_etf_50)

z_score_50['50 day Sortino Ratio (z score)'] = z_score_sortino_ratio_50
z_score_50['50 day Benchmark Minus ETF Sortino Ratio (z score)'] = z_score_benchmark_minus_etf_50
z_score_50.sort_values(by='50 day Sortino Ratio (z score)', ascending=True, inplace=True)

# Calculate z-scores for 200-day time frame
correlation_column = f'10 year Correlation to {benchmark}'
z_score_200['200 day Sortino Ratio (z score)'] = signal_labels.z_score(rolling_sortino_ratios_etf_200)
z_score_200['200 day Benchmark Minus ETF Sortino Ratio (z score)'] = signal_labels.z_score(rolling_sortino_ratios_benchmark_minus_etf_200)
z_score_200[correlation_column] = correlation_10y
z_score_200.sort_values(by='200 day Sortino Ratio (z score)', ascending=True, inplace=True)

# Calculate z-scores for 400-day time frame
z_score_sortino_ratio_400 = signal_labels.z_score(rolling_sortino_ratios_etf_400)
z_score_benchmark_minus_etf_400 = signal_labels.z_score(rolling_sortino_ratios_benchmark_minus_etf_400)

z_score_400['400 day Sortino Ratio (z score)'] = z_score_sortino_ratio_400
z_score_400['400 day Benchmark Minus ETF Sortino Ratio (z score)'] = z_score_benchmark_minus_etf_400
z_score_400.sort_values(by='400 day Sortino Ratio (z score)', ascending=True, inplace=True)

# Round decimal values
z_score_3 = z_score_3.round(2)
z_score_9 = z_score_9.round(2)
z_score_21 = z_score_21.round(2)
z_score_50 = z_score_50.round(2)
z_score_200 = z_score_200.round(2)
z_score_400 = z_score_400.round(2)

# Combine the longer-horizon columns into the primary table
z_score_combined = pd.concat([
    z_score_21,
    z_score_50,
    z_score_200.drop(columns=[correlation_column], errors='ignore'),
    z_score_400,
], axis=1)

# Build a complementary short-horizon / correlation table
z_score_complementary = pd.concat([
    z_score_21,
    z_score_9,
    z_score_3,
    z_score_200[[correlation_column]],
], axis=1)

# Sort columns by time frame and metric
#z_score_combined = z_score_combined.reindex(sorted(z_score_combined.columns, key=lambda x: (x.split()[0], x.split()[2])), axis=1)
#REVERSE ORDER OF COLUMNS (FROM RIGHT TO LEFT)
#CREATE FUNCTION THAT REINDEXES COLUMNS IN REVERSE ORDER
def reverse_column_order(df):
    df = df.reindex(columns=df.columns[::-1])
    return df

z_score_combined = reverse_column_order(z_score_combined)
z_score_complementary = z_score_complementary[[
    '21 day Sortino Ratio (z score)',
    '21 day Benchmark Minus ETF Sortino Ratio (z score)',
    '9 day Sortino Ratio (z score)',
    '9 day Benchmark Minus ETF Sortino Ratio (z score)',
    '3 day Sortino Ratio (z score)',
    '3 day Benchmark Minus ETF Sortino Ratio (z score)',
    correlation_column,
]]

# Plot the combined metrics
combined_metrics_fig = qp.plot_z_score_combined(z_score_combined)
combined_metrics_fig.update_layout(
    title='Combined Z-Scores for Sortino Ratios (21 / 50 / 200 / 400 Day)',
    autosize=True,
)
combined_metrics_fig.show(config={'responsive': True})

complementary_metrics_fig = qp.plot_z_score_combined(z_score_complementary)
complementary_metrics_fig.update_layout(
    title=f'Combined Z-Scores for Sortino Ratios (21 / 9 / 3 Day + 10 Year Correlation to {benchmark})',
    autosize=True,
)
complementary_metrics_fig.show(config={'responsive': True})
